# 05 · 장애 진단과 복구 확인

목표: 상태→영향→Trace→완화→재검증 순서로 장애 기록을 만든다. 약15분.

**이 파일의 핵심 질문:** 증상·근거·가설·조치·재검증을 구분했고, 실제로 복구된 범위는 어디까지인가?

**Liveness**는 프로세스 생존, **Readiness**는 요청 수신 준비 상태다. **Timeout**은 작업의 제한시간, **Mitigation**은 사용자 영향을 줄이는 완화 조치다. **Runbook**은 증상·조치·검증을 정리한 대응 절차다.

**수업 중 먼저 실행할 단계**

- 1 준비 상태와 생존 상태
- 2 시간 초과의 원인
- 3 실패 Trace 조회
- 4 복구 실험
- 6 상태 복원

파일 안의 변수 의존성이 있으므로 처음 실행할 때는 위에서부터 순서대로 진행한다. 핵심 단계에서는 출력을 읽고, 나머지 단계는 빠르게 실행한 뒤 복습 때 살펴본다.

## 시작 전 · 어디에서 무엇을 실행하나요?

1. VS Code에서 `llmops-16week` **폴더 전체**를 연다. 압축을 풀었다면 Notebook 파일만 따로 옮기지 않는다.
2. VS Code의 Python·Jupyter 확장을 설치하고 우측 위 **Select Kernel → Python Environments → .venv / Python 3.11**을 선택한다. 터미널의 가상환경과 Notebook 커널은 별도 선택이다.
3. 아래 **환경 확인 코드 셀부터** `Shift+Enter`로 한 셀씩 실행한다. 이 파일 안에서는 앞에서 만든 변수를 뒤에서 사용한다.
4. 각 단계의 **예상 결과**와 실제 출력을 대조한다. `assert`는 확인 조건이다. 출력이나 오류가 없어도 셀이 완료되면 조건을 통과한 것이다.
5. 셀을 건너뛰어 `NameError`가 나거나 변수를 여러 번 바꿨다면 **Restart Kernel → Run All**로 처음부터 실행한다. 다른 Notebook의 변수는 가져오지 않는다.

처음 환경을 만드는 경우에만 **VS Code 터미널**에서 실행한다. 아래 명령을 Python 코드 셀에 붙여넣지 않는다.

```bash
# macOS / Linux · 저장소 루트
python3.11 -m venv .venv
source .venv/bin/activate
python -m pip install -r requirements.txt
```

Windows PowerShell은 `py -3.11 -m venv .venv`, `.venv\Scripts\Activate.ps1` 순서다. 활성화가 제한된 환경에서는 `.venv\Scripts\python.exe -m pip install -r requirements.txt`처럼 가상환경 Python을 직접 실행한다. 이미 준비된 `.venv`는 그대로 쓴다.

**공통 실습의 실행 방식:** `lab_client`가 메모리 안에 FastAPI 앱을 만들고 시작·종료를 관리한다. HTTPX의 ASGITransport가 같은 프로세스에서 요청을 전달하므로 `uvicorn`, Docker, Ollama, API 키가 필요하지 않다. ASGI는 *Asynchronous Server Gateway Interface*, Python 비동기 웹 앱과 서버 사이의 인터페이스다.

Notebook 앱과 브라우저의 `127.0.0.1:8010` 앱은 **서로 다른 실행**이다. Notebook에서 만든 요청 수가 브라우저·Prometheus에 합쳐지지 않는다. 실제 모델 대신 고정 문장과 대기를 쓰므로 외부 비용은 발생하지 않는다. 실행 시간·요청 ID는 매번 달라져도 정상이다.

In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "week10/lab/app.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import asyncio, json
from dataclasses import replace
from week10.lab.config import Settings
from week10.lab.experiments import lab_client, fixture_check, window_delta, compare_releases
assert sys.version_info[:2] == (3, 11), "Select the Python 3.11 .venv kernel in VS Code"
print("Python:", sys.version.split()[0])
print("실행 환경:", sys.executable)
print("저장소:", ROOT)
print("환경 확인 완료 · 이 Notebook은 별도 서버 없이 실행합니다.")

## 1. 준비 상태와 생존 상태

의존성 장애에서 프로세스를 무조건 재시작하는 것이 적절한지 토론한다.

**예상 결과:** live200·ready503·새 assist503. 무출력 통과.

**결과 해석:** 프로세스는 살아 있지만 의존성 fixture가 준비되지 않았다. 새 요청을 받는 판단과 재시작 판단을 구분한다.

In [ ]:
async with lab_client() as (app, client):
    app.state.dependency_ok = False  # test fixture; production control API 아님
    live = await client.get("/health/live")
    ready = await client.get("/health/ready")
    blocked = await client.post("/api/v1/assist", json={"message":"예시"})
assert [live.status_code,ready.status_code,blocked.status_code] == [200,503,503]

## 2. 시간 초과의 원인

정상과 지연 조건을 한 앱에서 비교한다.

**예상 결과:** 정상200과 지연 조건504. 완료 후 active=0.

**결과 해석:** slow는 제한시간보다 긴 대기를 주입한다. 처리 슬롯이 해제되는지까지 확인한다.

In [ ]:
async with lab_client() as (app, client):
    normal = await client.post("/api/v1/assist", json={"message":"예시"})
    failed = await client.post("/api/v1/assist", json={"message":"예시","scenario":"slow"})
    snap = app.state.telemetry.snapshot()
assert failed.status_code == 504 and snap["active"] == 0

## 3. 실패 Trace 조회

HTTP 상태만으로 검색 장애·생성 지연을 같은 원인으로 결론내리지 않는다.

**예상 결과:** 실패한 요청의 retrieve OK·generate ERROR·assist ERROR. format 없음.

**결과 해석:** 시간 초과가 생성 단계에서 발생했음을 찾는다. 기기별 duration_ms 숫자 자체보다 실패 구간에 집중한다.

In [ ]:
tid = failed.headers["X-Trace-ID"]
fault_trace = [s for s in snap["spans"] if s["trace_id"] == tid]
[(s["name"],s["duration_ms"],s["status"]) for s in fault_trace]

## 4. 복구 실험

주입 조건을 normal로 돌리고 이후 요청의 성공과 active 해제를 함께 본다.

**예상 결과:** 후속 정상 요청200, active=0, 고정 정상 답변.

**결과 해석:** 앞 요청이 실패한 뒤에도 새 요청을 처리할 수 있는지 확인한다. 새 lab_client 블록은 이전 실험과 별도 앱이다.

In [ ]:
async with lab_client() as (app, client):
    await client.post("/api/v1/assist", json={"message":"예시","scenario":"slow"})
    recovered = await client.post("/api/v1/assist", json={"message":"예시"})
    assert recovered.status_code == 200 and app.state.telemetry.active == 0
    print(recovered.json())

## 5. 수신 중단과 처리 종료

이 앱은 비스트리밍이다. 연결 종료만으로 외부 모델 계산이 멈춘다고 가정하지 않는다.

**예상 결과:** 고정 문장+asyncio.sleep, 외부 GPU 취소 미구현·미측정 안내.

**결과 해석:** 로컬 대기 취소가 실제 모델 서버의 연산 취소를 증명하지 않는다. 운영에서는 제공자의 취소·과금 조건을 별도 검증한다.

In [ ]:
print("이 실험의 제공자: 고정 문장+asyncio.sleep")
print("외부 GPU 취소: 미구현·미측정")

## 6. 상태 복원

의존성 fixture 복원 후 ready·정상 응답·기준 통과를 확인한다.

**예상 결과:** 의존성 복원 뒤 ready200·정상 응답·fixture 통과. 무출력 통과.

**결과 해석:** 상태 검사와 내용 검사까지 재확인해야 복구 근거가 된다. dependency_ok는 교육용 내부 조작이다.

In [ ]:
async with lab_client() as (app, client):
    app.state.dependency_ok = False
    assert (await client.get("/health/ready")).status_code == 503
    app.state.dependency_ok = True
    assert (await client.get("/health/ready")).status_code == 200
    r = await client.post("/api/v1/assist", json={"message":"예시"})
    assert fixture_check(r.json()["answer"])

## 7. 장애 기록

실측·가설·실험실 조작을 구분한다. 재시작은 영구 해결의 증거가 아니다.

**예상 결과:** 증상·Trace 근거·가설·완화·재검증·남은 위험을 담은 사전.

**결과 해석:** 이 예시는 코드가 채운 기록이다. 자신의 관찰 결과로 고쳐 쓰고 미검증 범위를 남긴다.

In [ ]:
incident = {"symptom":"504", "evidence_trace":tid, "hypothesis":"generate 지연", "mitigation":"주입 조건 normal 복원", "verification":"후속200 + 기준통과 + active0", "remaining":"실제 제공자/장기 관찰 미검증"}
incident

## 선택 확장 · 실제 서비스 화면에서 장애·복구 확인

앞의 Notebook 실험을 마친 뒤 진행한다. **VS Code 터미널**을 열고 저장소 루트에서 기존 `.venv`를 활성화한다. Docker 경로의 API가 이미8010포트에서 실행 중이면 그 앱 화면을 사용하고 서버를 중복 실행하지 않는다.

```bash
python -m uvicorn week10.lab.app:app --host 127.0.0.1 --port 8010 --no-access-log
```

1. 웹 브라우저에서 [실습 앱](http://127.0.0.1:8010/)을 연다. 실행 중인 버전이v1인지 확인한다.
2. 정상 처리: HTTP200·simulated=true, retrieve·generate·format의 정상 실행 확인.
3. 검색 단계 오류: HTTP502, retrieve ERROR와 generate 미실행 확인.
4. 제한시간 초과: HTTP504, generate ERROR 확인.
5. 다시 정상 처리: HTTP200 확인. [snapshot](http://127.0.0.1:8010/lab/snapshot)에서 active=0 확인.
6. 요청 ID·Trace ID로 같은 실행의 로그와 Span을 연결한다. [지표 텍스트](http://127.0.0.1:8010/metrics)에서 상태별 누적 건수를 읽는다.
7. 자신의 서버 터미널에서 Ctrl+C로 종료한다. 재시작하면 메모리 로그·지표·Span은 초기화된다.

각 조건의 화면1개와 실패 구간·복구 근거를 기록한다. 이 앱과 Notebook은 별도 실행이므로 건수·ID가 같을 필요는 없다. port already in use이면 자신이 실행한 서버를 확인하거나 `--port 8011`로 바꾸고 브라우저 주소도 변경한다.

Windows PowerShell의 환경 활성화는 `.venv\Scripts\Activate.ps1`, Mac/Linux는 `source .venv/bin/activate`다. v2로 변경했던 터미널이라면 서버 시작 전에 v1으로 복원한다. PowerShell은 `$env:WEEK10_RELEASE="v1"`, Mac/Linux는 `export WEEK10_RELEASE=v1`이다.

## 직접 바꿔볼 실험 · 선택 복습

1. 시간 초과504와 검색 오류502에서 실패 Span을 비교한다.
2. 의존성 복원 전후에 live·ready 상태를 각각 기록한다.
3. ‘서버 재시작’만 적은 복구 기록에 정상 응답·내용 검사·자원 해제 근거를 보완한다.

한 번에 한 조건만 바꾸고 원래 코드와 비교한다. 기대값이 고정된 assert가 있다면 새 실험 조건에 맞는 값도 함께 설명한다.

## 프로젝트 적용 기록

**정리 질문:** 증상·근거·가설·조치·재검증을 구분했고, 실제로 복구된 범위는 어디까지인가?

아래 Markdown 셀을 더블클릭해 자신의 결과를 적는다. 다른 파일 제출이나 화면 실행을 자동으로 수행하지 않는다.

| 항목 | 직접 기록 |
|---|---|
| 예상 | 실행 전 예상한 결과 |
| 관찰 | 상태 코드·건수·실패 단계 등 실제 결과 |
| 근거 | 실행한 단계 번호·변경한 조건·캡처 위치 |
| 판단 | 유지·수정·채택·보류 이유 |
| 미검증·다음 변경 | 아직 실행하지 않은 조건과 다음 비교 |

학생 정보나 원문 입력은 공개 저장소에 올리지 않는다. 실제 실행하지 않은 조건은 미측정으로 표시한다.

## 막혔을 때의 확인 순서

| 증상 | 확인·해결 |
|---|---|
| 커널 선택 창만 표시 | Python·Jupyter 확장 설치 후 `.venv`의 Python 선택 |
| `ModuleNotFoundError` | 커널 경로 확인 → 해당 `.venv`로 `python -m pip install -r requirements.txt` |
| 환경 확인에서 저장소를 찾지 못함 | 저장소 전체를 열고 원래 `week10/lab/notebooks` 위치에서 실행 |
| `NameError` | 앞 셀 실행 여부 확인 → Restart Kernel → Run All |
| `asyncio.run() cannot be called...` | Jupyter에서는 제공된 최상위 `await`·`async with`를 그대로 사용 |
| 셀에 출력이 없음 | `assert`만 있는 셀은 오류 없이 끝나면 통과. 왼쪽 실행 번호와 완료 표시 확인 |
| 예상한 422·502·503·504 | 입력 오류·검색 오류·준비 실패·시간 초과를 의도적으로 만든 단계인지 확인 |
| 예상과 다른 `AssertionError` | 바로 위 코드·출력 확인. 변경한 값을 복원하고 처음부터 다시 실행. assert를 지워 통과시키지 않음 |
| 실행 시간·ID가 예시와 다름 | 고정값 비교 대신 상태·부모/자식 관계·분모·검증 결과를 비교 |

실습에서 바꾼 내용은 자신의 사본에 기록한다. 공개 저장소에는 학생 정보·원문 질문·실제 키를 올리지 않는다.